# BıRAG `v0.1.1` — Colab + Unsloth eğitim ve üretim (ön kayıt EK-2)

Bu defter `v0.1.1` ön kaydının **Faz 4**'ünü koşar: iki kol × 8 tohum eğitim, ardından 6 eksen + çok turlu üretim.

| kol | veri | tarif |
|---|---|---|
| `v010u` | `datasets/v0.1.0` (= `v0.0.22`) | `configs/training/e3-celiskili-k8qo-v022-t{t}.yaml` |
| `v011u` | `datasets/v0.1.1` | `configs/training/v011-k8qo-t{t}.yaml` |

⛔ **Kurallar (EK-2 — `configs/deney/2026-09-29-v011-on-kayit-ek2.json`):**
- Bu defter **mühürlü**. Değiştirmeyin. Değişiklik gerekiyorsa **ilk çıktıdan önce** EK-3 yazılır.
- İki kol **aynı ortamda**: kütüphane sürümleri, model revizyonu, şablon ve LoRA yolu 16 koşunun hepsinde aynı olmak zorunda (defter ilk oturumda referans yazar, sonra denetler).
- **bf16, 4-bit YOK** (K7 · `e3` bf16). GPU ≥ 22 GB (A100 / L4). T4 yetmez.
- Sıra tohum içinde dönüşümlü: `v010u-t7`, `v011u-t7`, `v010u-t13`, … — oturum değişirse iki kolu birlikte etkiler.
- Her adım **yeniden başlatmaya dayanıklı**: çıktılar Drive'a yazılır, biten adım atlanır, yarıda kalan eğitim son kontrol noktasından sürer.
- Puanlama **burada yapılmaz**: ham üretim Drive'a yazılır, puanlama ve hüküm veri makinesinde.

Hücreleri sırayla koşun. Oturum koparsa baştan (1. hücreden) yeniden koşun — biten iş atlanır.

## 1. Parametreler

In [ ]:
MODEL_ID = "unsloth/gemma-4-E4B-it"   # ⚠️ DOĞRULAYIN: `e3`'ün tabanı mlx-community/gemma-4-E4B-it-bf16 idi (google/gemma-4-E4B-it'in MLX kopyası).
                                      #    Unsloth kopyası ya da google/… aslı olabilir — 16 koşuda AYNI olmalı.
MODEL_REVIZYON = None                 # None: ilk oturumda çözülür ve referansa yazılır; sonraki oturumlar referanstan okur
LORA_YOLU = "unsloth"                 # "unsloth" | "peft". Kurulum sınaması (3. adım) LoRA denetiminde durursa "peft" yapın
                                      #    — ama YALNIZ hiç eğitim yapılmadan önce (ortam referansı değişikliği durdurur)
DRIVE_KOK = "/content/drive/MyDrive/birag-v011"
REPO = "github.com/enveryildirim/birag-data-generation.git"
REPO_DIZIN = "/content/birag-data-generation"

## 2. Drive, depo, kurulum

Depo özelse Colab **Gizli Anahtarlar**'a (🔑) `GITHUB_TOKEN` ekleyin; belirteç URL'ye yazılmaz, yalnız bu komut için başlık olarak geçer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64, os, subprocess
def _git(*a):
    ek = []
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t:
            b = base64.b64encode(f"x-access-token:{t}".encode()).decode()
            ek = ["-c", f"http.extraHeader=Authorization: Basic {b}"]
    except Exception:
        pass
    subprocess.run(["git", *ek, *a], check=True)

if os.path.exists(REPO_DIZIN):
    _git("-C", REPO_DIZIN, "pull", "--ff-only")
else:
    _git("clone", f"https://{REPO}", REPO_DIZIN)
print(subprocess.run(["git", "-C", REPO_DIZIN, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q unsloth
# ⚠️ Kurulum torch'u güncellediyse Colab «oturumu yeniden başlat» isteyebilir: yeniden başlatıp 1. hücreden devam edin.

## 3. Kurulum sınaması — hiçbir eğitimden önce

Mühür · GPU · şablon (35 referans dizge) · jeton uzunluğu (ön kaydın adım 2'si) · LoRA hedefleri. Biri tutmazsa **durur**.

In [ ]:
import unsloth  # noqa: F401  — transformers'tan ÖNCE
import sys, json, torch
from pathlib import Path
sys.path.insert(0, f"{REPO_DIZIN}/src")
import colab_egitim as ce

CIKTI = Path(DRIVE_KOK); CIKTI.mkdir(parents=True, exist_ok=True)
REFERANS = CIKTI / "ortam-referans.json"

print("mühürlü dosya:", ce.muhur_denetle(), "— hepsi mühürle aynı")
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb >= 22, f"⛔ GPU {gb:.0f} GB — bf16 E4B için A100/L4 gerekli; 4-bit bir sapmadır (EK-3 gerekir)"
print("GPU:", torch.cuda.get_device_name(0), f"{gb:.0f} GB")

if REFERANS.exists():
    REV = json.loads(REFERANS.read_text())["model_revizyon"]
else:
    from huggingface_hub import model_info
    REV = MODEL_REVIZYON or model_info(MODEL_ID).sha
git_rev = subprocess.run(["git", "-C", REPO_DIZIN, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
ORTAM = ce.ortam(MODEL_ID, REV, LORA_YOLU, git_rev)
yukle = ce.unsloth_yukle(MODEL_ID, REV)

model, tokenizer = yukle(2048)
print("şablon:", ce.sablon_referansi_denetle(tokenizer), "dizge referansla birebir")
tok = ce.sablon_kur(tokenizer)
for kol in ce.KOLLAR:
    print("uzunluk:", ce.uzunluk_denetle(tok, kol))          # > 2048 ise DURUR (sessiz kırpma kıyası bozar)
c = ce.tarif("v011u", ce.TOHUM[0])
hedef, n_katman = ce.lora_hedefleri(model, c["ust_katman"], c["anahtarlar"])
print(f"dil katmanı {n_katman} · LoRA hedefi {len(hedef)}: {hedef[0]} … {hedef[-1]}")
beklenen = ce.lora_parametre_beklenen(model, hedef, c["r"])
pm = ce.lora_uygula(model, c, hedef, ce.TOHUM[0], LORA_YOLU)
print("LoRA denetimi:", ce.lora_denetle(pm, hedef, beklenen))
del pm, model; torch.cuda.empty_cache()

print("ortam:", ce.ortam_denetle(ORTAM, REFERANS))

## 4. Eğitim — 2 kol × 8 tohum

Her koşu tabanı yeniden yükler (temiz başlangıç). Adapter varsa atlanır; yarıda kalmışsa son kontrol noktasından sürer.

In [ ]:
import time
for t in ce.TOHUM:
    for kol in ("v010u", "v011u"):
        t0 = time.time()
        k = ce.egit(kol, t, yukle, CIKTI, LORA_YOLU, ORTAM)
        if k.get("atlandi"):
            print(f"  {kol}-t{t}: zaten eğitilmiş")
        else:
            son = [x for x in k["log"] if "eval_loss" in x]
            print(f"✅ {kol}-t{t}: {k['sure_dk']} dk · sürdü={k['surdu']} · son eval_loss {son[-1]['eval_loss'] if son else '—'}")

## 5. Üretim — 6 eksen + çok turlu, iki kol

`thinking` kapalı · 1024 jeton · açgözlü (sıcaklık 0) — `e3`'ün kayıtlı koşularıyla aynı ayar. Biten hücre atlanır.

In [ ]:
def _bitti(kol, t):
    ek = all(ce.bitmis_mi(CIKTI / "eksen-kosu", kol, t, kisa, sum(1 for x in open(f"{REPO_DIZIN}/{yol}") if x.strip()))
             for kisa, yol in ce.EKSEN)
    ct = ce.bitmis_mi(CIKTI / "cok-turlu-kosu", kol, t, "cokturlu", 40)
    return ek and ct

for t in ce.TOHUM:
    for kol in ("v010u", "v011u"):
        if _bitti(kol, t):
            print(f"  {kol}-t{t}: üretim zaten tam"); continue
        adapter = CIKTI / "runs" / f"{kol}-t{t}" / "adapter"
        assert (adapter / "adapter_config.json").exists(), f"⛔ {kol}-t{t} eğitilmemiş — önce 4. adım"
        print("ortam:", ce.ortam_denetle(ORTAM, REFERANS))
        model, tok = ce.adapter_yukle(yukle, adapter, 2048)
        for kisa, yol in ce.EKSEN:
            d = ce.eksen_kos(model, tok, kol, t, kisa, yol, CIKTI, ORTAM, str(adapter)); print("  ", d.name)
        d = ce.cok_turlu_kos(model, tok, kol, t, CIKTI, ORTAM, str(adapter)); print("  ", d.name)
        del model; torch.cuda.empty_cache()

## 6. Durum ve paket

In [ ]:
import shutil, zipfile
tam = sum(_bitti(k, t) for t in ce.TOHUM for k in ("v010u", "v011u"))
egitilen = sum((CIKTI / "runs" / f"{k}-t{t}" / "adapter" / "adapter_config.json").exists() for t in ce.TOHUM for k in ("v010u", "v011u"))
print(f"eğitim {egitilen}/16 · üretim {tam}/16 (her biri 6 eksen + çok turlu)")
if tam == 16:
    paket = CIKTI / "v011-colab-ciktilari.zip"
    with zipfile.ZipFile(paket, "w", zipfile.ZIP_DEFLATED) as z:
        for alt in ("eksen-kosu", "cok-turlu-kosu"):
            for f in (CIKTI / alt).rglob("*"):
                if f.is_file(): z.write(f, f.relative_to(CIKTI))
        for f in (CIKTI / "runs").glob("*/egitim.json"): z.write(f, f.relative_to(CIKTI))
        z.write(REFERANS, REFERANS.relative_to(CIKTI))
    print("✅ paket:", paket)

## 7. Veri makinesine dönüş

1. `v011-colab-ciktilari.zip`'i indirin; içindeki `eksen-kosu/` ve `cok-turlu-kosu/` dizinlerini depoda `reports/analiz/` altına açın.
2. Puanlama (üretim koşmaz):  `uv run python scripts/analiz/2026-09-29-v011-colab-puanla.py`
3. Hüküm:  `uv run python scripts/analiz/2026-09-29-v011-onkayit-ek2-cozumleme.py` → `reports/analiz/2026-09-29-v011-onkayit-ek2-sonuc.md`

Çözümleme mühür, tamlık (2 kol × 8 tohum × 7 hücre), ayar ve ortam kilitleri geçmeden **puan okumaz**.